# M07-01 — Parquet y layout analítico

[← Anterior](01-teoria.ipynb) · [Siguiente →](../../README.md)

Este fichero es el **guion**. No lo rellenes aquí: **crea tu propio notebook** y ve construyéndolo celda a celda.

## Qué vas a hacer

Publicar `data/curated/sales_analytics` en Parquet particionado por mes y demostrar que un filtro de mes no lee el año entero.

## 0 — Crea tu notebook

1. En el explorador, abre la carpeta `notebooks/trabajo/`.
2. Clic derecho → **New File…**
3. Nombre exacto: `M07-01-parquet-layout.ipynb` (incluye `.ipynb`).
4. Ábrelo. Arriba a la derecha (o `F1` → `Notebook: Select Notebook Kernel`) elige **Python (NovaShop)**.
5. Deja **este** guion a un lado (pestaña) y escribe **solo** en el tuyo.

## Cómo organizar *tu* notebook (siempre)

En cada paso creas **dos celdas**, en este orden:

1. **Markdown** — qué vas a hacer y por qué, con tus palabras.
2. **Código** — el de la celda de código del paso. Lo ejecutas (`Shift+Enter`), miras la salida y, si no cuadra, lo mejoras.

No dejes un muro de código sin explicación. Un notebook se lee de arriba abajo, como un cuaderno.

> Kernel **Python (NovaShop)**. Si no aparece: terminal → `bash .devcontainer/setup.sh` → vuelve a elegir kernel.


### Paso 1 — Dataset curated

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Left al catálogo conserva P999. Inner a clientes quita CX*. Solo paid. dropDuplicates en product_id.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** **1122** filas (mismo universo que M04-02).

**Por qué este paso.** Si sale 2244, el catálogo no era único.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


from pyspark.sql.functions import col

spark = get_spark("novashop-m07")
fact = spark.read.parquet(str(STAGING / "fact_lines"))
customers = spark.read.parquet(str(STAGING / "customers_clean"))
products = (
    spark.read.parquet(str(STAGING / "products_clean"))
    .dropDuplicates(["product_id"])
)
sales = (
    fact.join(customers, "customer_id", "inner")
    .join(products, "product_id", "left")
    .where(col("is_billable"))
    .select(
        "order_id", "order_ts", "order_month",
        "customer_id", "country", "segment",
        "product_id", "category",
        "qty", "unit_price", "discount", "gmv_line", "channel_norm",
    )
)
print(sales.count())


### Paso 2 — Escribe Parquet por mes

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

overwrite deja el curated idempotente. Doce particiones = doce meses de 2024.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Carpetas `order_month=2024-01` … `order_month=2024-12` (más `_SUCCESS`).

**Por qué este paso.** Esto es layout de disco, no el repartition de M06.


In [ ]:
dest = CURATED / "sales_analytics"
CURATED.mkdir(parents=True, exist_ok=True)
(
    sales.write.mode("overwrite")
    .partitionBy("order_month")
    .parquet(str(dest))
)
print(sorted(p.name for p in dest.iterdir() if p.is_dir()))


### Paso 3 — Prune al leer un mes

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

El plan debe listar solo marzo (o PartitionFilters: order_month=2024-03).

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Total **1122**. `marzo` es un subconjunto. El formatted menciona `2024-03`.

**Por qué este paso.** Copia en Markdown la línea del PartitionFilters.


In [ ]:
marzo = spark.read.parquet(str(dest)).where(col("order_month") == "2024-03")
marzo.explain("formatted")
print("marzo", marzo.count(), "total", spark.read.parquet(str(dest)).count())


### Paso 4 — CSV vs Parquet (schema, no solo tamaño)

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

coalesce(1) solo existe aquí para comparar *un* CSV, no como patrón. Releo los dos schemas.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Parquet mantiene `decimal`/`timestamp`. El CSV vuelve a string. El tamaño: Parquet suele ganar; en este volumen a veces es parecido.

**Por qué este paso.** Curated en CSV “para el analista” pierde tipos.


In [ ]:
import os

csv_dir = CURATED / "_csv_compare"
sales.coalesce(1).write.mode("overwrite").option("header", True).csv(str(csv_dir))

def du(path):
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file())

print("parquet", du(dest), "csv", du(csv_dir))
spark.read.parquet(str(dest)).printSchema()
spark.read.option("header", True).csv(str(csv_dir)).printSchema()


## Comprueba

Antes de dar el lab por cerrado, vuelve a ejecutar de arriba abajo (**Run All**) y verifica:

Vuelve a ejecutar el `write.mode("overwrite")` y cuenta.
Sigue **1122**. No se duplica. Anótalo.


## Mejora — Dos claves de partición

Copia `sales_analytics_geo` con `partitionBy("order_month", "country")` y lee marzo ∧ ES. No particiones por customer_id.

Si te atasca, el código está en la celda siguiente.


In [ ]:
geo = CURATED / "sales_analytics_geo"
sales.write.mode("overwrite").partitionBy("order_month", "country").parquet(str(geo))
(
    spark.read.parquet(str(geo))
    .where((col("order_month") == "2024-03") & (col("country") == "ES"))
    .explain("formatted")
)


## Si algo falla

| Qué ves | Suele ser | Qué haces |
|---------|-----------|-----------|
| Miles de part-000xx | repartition(200) residual | repartition(12, order_month) antes del write |
| Count 2244 | append o join duplicado | overwrite + dropDuplicates de productos |
| order_month no está al leer | API antigua | spark.read.parquet de 3.5 sí la incluye |


## Siguiente

Cuando hayas **comprobado** y (si quieres) **mejorado**, abre [índice del curso](../../README.md).
